# Olist Review-Analysis Pipeline — MLWB entry point

This notebook is the entry point the Celonis Action Flow executes. The first code
cell is tagged `parameters` (View → Cell Toolbar → Tags) so the Action Flow can
overwrite its values at runtime. All real logic lives in `src/`.

**Before committing:** Kernel → Restart & Clear Output (or
`jupyter nbconvert --clear-output --inplace pipeline.ipynb`) to avoid leaking data.

In [ ]:
# %pip install -r requirements.txt

In [ ]:
#%pip install pycelonis_llm

In [ ]:
import os
os.makedirs("test-data", exist_ok=True)
csv = """review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
syn0001,ord0001,5,,"Produto excelente, chegou antes do prazo. Recomendo!",2018-01-01 00:00:00,2018-01-02 10:00:00
syn0002,ord0002,1,,O produto chegou quebrado e a entrega atrasou muito.,2018-01-03 00:00:00,2018-01-04 10:00:00
syn0003,ord0003,3,,,2018-01-05 00:00:00,2018-01-06 10:00:00
syn0004,ord0004,2,atraso,Comprei pelo prazo de entrega e ate agora nao chegou.,2018-01-07 00:00:00,2018-01-08 10:00:00
syn0005,ord0005,4,,O preco estava otimo mas a embalagem veio amassada.,2018-01-09 00:00:00,2018-01-10 10:00:00
syn0006,ord0006,5,,"Atendimento ao cliente foi muito atencioso, resolveram meu problema.",2018-01-11 00:00:00,2018-01-12 10:00:00
syn0007,ord0007,1,,"   ",2018-01-13 00:00:00,2018-01-14 10:00:00
syn0008,ord0008,5,,"Muito caro pelo que oferece, mas o produto funciona bem.",2018-01-15 00:00:00,2018-01-16 10:00:00
syn0009,ord0009,1,reembolso,"Veio o produto errado e quero o reembolso imediato, ninguem responde!",2018-01-17 00:00:00,2018-01-18 10:00:00
syn0010,ord0010,5,,"Chegou rapido, embalagem perfeita, recomendo a loja para todos!",2018-01-19 00:00:00,2018-01-20 10:00:00
"""
with open("test-data/sample_reviews.csv", "w") as f:
    f.write(csv)
print("wrote", os.path.getsize("test-data/sample_reviews.csv"), "bytes")

In [ ]:
%%writefile .env
aws_access_key_id = "BOZDOJIYJNSTXHDEISVS"
aws_secret_access_key = "CFEEYNONRUDLOUKKSCLSAJCRIAFBZKBSKFKKTQLT"

In [ ]:
from pycelonis import get_celonis
from pycelonis_llm.llm import LLM
print([m.id for m in LLM(get_celonis().client).get_models()])

In [ ]:
# Cell tagged "parameters" — the Action Flow overwrites these at runtime.
input_filename = None   # CSV delivered into input-data/; None -> local sample
max_rows = 3         # optional cap for cheap runs

In [ ]:
# Main execution
import os
import sys
from dotenv import load_dotenv

load_dotenv()

# In a notebook there is no __file__, so add the current working directory to the path.
sys.path.insert(0, os.path.abspath(""))

from src.pipeline import run
from src.ingestion import push_to_celonis

result = run(input_filename=input_filename, max_rows=max_rows)
print(f"Analysed {len(result)} reviews")
result.head()

In [ ]:
# Persist results. Honors ingestion.write_back in config.yaml:
#   false -> writes output/review_insights.parquet locally (S3 skipped)
#   true  -> pushes to the Celonis data pool via the S3 ingestion API
destination = push_to_celonis(result)
print(f"Done: {destination}")